# Dataset 3 — Qwen Fine-Tuning: Balanced Binary Pairs (Per Dataset)

Builds **two separate** fine-tuning datasets — one for Wikidata, one for DBpedia —
each in the format:
```
{ "anchor_query": str, "query": str, "label": 0 | 1 }
```
**label=1** → `query` is a positive (semantically similar) to `anchor_query`  
**label=0** → `query` is a negative (semantically dissimilar)

### Why separate datasets?
- Wikidata and DBpedia have different SPARQL dialects, predicates, and ontologies
- Each fine-tuned Qwen model specialises in one knowledge graph
- Cleaner evaluation: model performance is directly comparable per KG

### Output structure
```
qwen_finetune_dataset/
  wikidata/
    train.csv / train.json / train.jsonl
    val.csv   / val.json   / val.jsonl
    DATASET_CARD.txt
  dbpedia/
    train.csv / train.json / train.jsonl
    val.csv   / val.json   / val.jsonl
    DATASET_CARD.txt
```

### Balancing strategy (per anchor)
1. Cap positives at `MAX_POS_PER_ANCHOR=10` (random sample if more)
2. Fill negatives to match n_pos: **hard negatives first** (TF-IDF mined), then **cross-anchor fill**
3. If still insufficient, reduce positives to match available negatives
4. Stratified 90/10 train-val split (seed=42)

In [ ]:
import json
import random
import hashlib
import pandas as pd
import numpy as np
from pathlib import Path
from urllib.request import urlopen
from collections import defaultdict
from sklearn.model_selection import train_test_split

# ── Configuration ─────────────────────────────────────────────────────────
SEED               = 42
MAX_POS_PER_ANCHOR = 10    # cap positives per anchor to avoid dominance
VAL_RATIO          = 0.10  # 10% held out as validation
OUTPUT_ROOT        = Path("qwen_finetune_dataset")

BASE = "dataset"
SOURCES = {
    "wikidata": f"{BASE}/hard_triplets_tfidf_wikidata.json",
    "dbpedia" : f"{BASE}/hard_triplets_tfidf_dbpedia.json",
}

# Create per-dataset output directories
for ds in SOURCES:
    (OUTPUT_ROOT / ds).mkdir(parents=True, exist_ok=True)

rng = random.Random(SEED)
np.random.seed(SEED)
print("✅ Config ready. Output folders:")
for ds in SOURCES:
    print(f"  {OUTPUT_ROOT / ds}")

In [ ]:
def load_json_url(url):
    with urlopen(url) as r:
        return json.loads(r.read().decode())

raw_data = {}
for ds_name, url in SOURCES.items():
    data = load_json_url(url)
    raw_data[ds_name] = data
    n_pos = sum(len(r.get("positive_queries", r.get("positives", []))) for r in data)
    n_neg = sum(len(r.get("negative_queries", r.get("negatives", r.get("hard_negatives", [])))) for r in data)
    print(f"{ds_name:10s} | {len(data):4d} anchors | {n_pos:6d} positives | {n_neg:6d} hard-negatives")

print("\n✅ Data loaded.")

In [ ]:
def get_field(record, *keys):
    for k in keys:
        if k in record:
            return record[k]
    raise KeyError(f"None of {keys} found in {list(record.keys())}")

def to_str(item):
    if isinstance(item, str):  return item.strip()
    if isinstance(item, dict): return get_field(item, "query","sparql","sparqlQuery","text").strip()
    raise TypeError(f"Unexpected type: {type(item)}")

def parse_records(records):
    """Normalise raw JSON records into clean dicts."""
    parsed = []
    for rec in records:
        anchor    = to_str(get_field(rec, "anchor_query", "anchor"))
        pos_raw   = rec.get("positive_queries", rec.get("positives", rec.get("positive", [])))
        neg_raw   = rec.get("negative_queries", rec.get("negatives",
                    rec.get("negative", rec.get("hard_negatives", []))))
        positives = list(dict.fromkeys(to_str(p) for p in pos_raw))
        neg_full  = [to_str(n) for n in neg_raw]
        pos_set   = set(positives)
        negatives = list(dict.fromkeys(n for n in neg_full if n not in pos_set and n != anchor))
        if positives:
            parsed.append({"anchor": anchor, "positives": positives, "negatives": negatives})
    return parsed

# ── Inspect one sample record ─────────────────────────────────────────────
sample_ds   = "wikidata"
sample_rec  = raw_data[sample_ds][0]
print("Sample record keys:", list(sample_rec.keys()))
parsed_sample = parse_records([sample_rec])[0]
print(f"Anchor   : {parsed_sample['anchor'][:80]}...")
print(f"Positives: {len(parsed_sample['positives'])}")
print(f"Negatives: {len(parsed_sample['negatives'])}")

In [ ]:
def build_balanced_pairs(parsed_records, seed=SEED, max_pos=MAX_POS_PER_ANCHOR):
    """
    Given parsed anchor records for ONE dataset, produce a balanced list of
    (anchor_query, query, label) dicts.

    Balancing:
      - n_pos = min(len(positives), max_pos)
      - n_neg = n_pos  (hard negatives first, cross-anchor fill if needed)
    """
    local_rng = random.Random(seed)

    # Build cross-anchor negative pool (all positives from other anchors)
    cross_pool = []   # list of (anchor, query) so we can exclude own anchor
    for rec in parsed_records:
        for q in rec["positives"]:
            cross_pool.append((rec["anchor"], q))

    pairs  = []
    stats  = {"pos": 0, "neg_hard": 0, "neg_cross": 0, "reduced": 0}

    for rec in parsed_records:
        anchor    = rec["anchor"]
        positives = rec["positives"][:]
        negatives = rec["negatives"][:]
        all_pos   = set(positives)  # full set before capping

        # Step 1: cap positives
        if len(positives) > max_pos:
            positives = local_rng.sample(positives, max_pos)
        n_pos = len(positives)

        # Step 2a: hard negatives
        local_rng.shuffle(negatives)
        selected_negs = []
        for n in negatives:
            if len(selected_negs) >= n_pos: break
            if n not in all_pos and n != anchor:
                selected_negs.append((n, "hard"))

        # Step 2b: cross-anchor fill
        deficit = n_pos - len(selected_negs)
        if deficit > 0:
            candidates = list(dict.fromkeys(
                q for (a, q) in cross_pool
                if a != anchor and q not in all_pos and q != anchor
            ))
            local_rng.shuffle(candidates)
            for q in candidates[:deficit]:
                selected_negs.append((q, "cross"))

        # Step 3: reduce positives if still not enough negatives
        if len(selected_negs) < n_pos:
            positives = positives[:len(selected_negs)]
            stats["reduced"] += 1

        n_final = len(positives)
        selected_negs = selected_negs[:n_final]

        for q in positives:
            pairs.append({"anchor_query": anchor, "query": q, "label": 1, "neg_source": "positive"})
            stats["pos"] += 1

        for q, src in selected_negs:
            pairs.append({"anchor_query": anchor, "query": q, "label": 0, "neg_source": src})
            if src == "hard": stats["neg_hard"] += 1
            else:             stats["neg_cross"] += 1

    return pairs, stats

print("✅ Balancing function defined.")

In [ ]:
QWEN_COLS = ["anchor_query", "query", "label"]

def make_pair_id(anchor, query, label):
    return hashlib.md5(f"{anchor}|||{query}|||{label}".encode()).hexdigest()[:16]

def save_dataset(df_train, df_val, out_dir, ds_name, stats):
    out_dir = Path(out_dir)

    for split_name, split_df in [("train", df_train), ("val", df_val)]:
        # CSV
        split_df.to_csv(out_dir / f"{split_name}.csv", index=False)
        # JSON
        split_df[QWEN_COLS].to_json(out_dir / f"{split_name}.json",
                                     orient="records", indent=2, force_ascii=False)
        # JSONL
        with open(out_dir / f"{split_name}.jsonl", "w", encoding="utf-8") as f:
            for _, row in split_df[QWEN_COLS].iterrows():
                f.write(json.dumps({"anchor_query": row.anchor_query,
                                    "query": row.query,
                                    "label": int(row.label)},
                                   ensure_ascii=False) + "\n")

    df_all = pd.concat([df_train, df_val])
    card = f"""
================================================================
  Qwen Fine-Tuning Dataset — {ds_name.upper()}
================================================================
Format       : anchor_query | query | label (0 or 1)
Total pairs  : {len(df_all)}
  Train      : {len(df_train)}  |  Val: {len(df_val)}
Balance      : {(df_all['label']==1).sum()} positive / {(df_all['label']==0).sum()} negative

Negative composition
  Hard (TF-IDF mined) : {stats['neg_hard']}
  Cross-anchor fill   : {stats['neg_cross']}
  Anchors reduced     : {stats['reduced']}

Balancing strategy
  Max {MAX_POS_PER_ANCHOR} positives per anchor
  Equal negatives per anchor (hard first, cross-anchor fill)
  Stratified 90/10 train-val split (seed={SEED})

Files
  train.csv / train.json / train.jsonl
  val.csv   / val.json   / val.jsonl
================================================================
"""
    with open(out_dir / "DATASET_CARD.txt", "w") as f:
        f.write(card)
    print(card)

print("✅ Save function defined.")

In [ ]:
print("=" * 55)
print("  Processing: WIKIDATA")
print("=" * 55)

parsed_wikidata = parse_records(raw_data["wikidata"])
print(f"Parsed {len(parsed_wikidata)} anchors with ≥1 positive")

pairs_wd, stats_wd = build_balanced_pairs(parsed_wikidata)

df_wd = pd.DataFrame(pairs_wd)
df_wd["pair_id"] = df_wd.apply(
    lambda r: make_pair_id(r.anchor_query, r.query, r.label), axis=1)
df_wd = df_wd.sample(frac=1, random_state=SEED).reset_index(drop=True)

df_wd_train, df_wd_val = train_test_split(
    df_wd, test_size=VAL_RATIO, random_state=SEED, stratify=df_wd["label"])

print(f"\nTotal: {len(df_wd)} pairs")
print(f"Train: {len(df_wd_train)}  |  Val: {len(df_wd_val)}")
print("Label distribution (train):")
print(df_wd_train["label"].value_counts().rename({1:"positive(1)", 0:"negative(0)"})
      .to_string())
print("\nNegative sources:")
print(df_wd[df_wd.label==0]["neg_source"].value_counts().to_string())

In [ ]:
same   = (df_wd["anchor_query"] == df_wd["query"]).sum()
pos_s  = set(zip(df_wd[df_wd.label==1].anchor_query, df_wd[df_wd.label==1].query))
neg_s  = set(zip(df_wd[df_wd.label==0].anchor_query, df_wd[df_wd.label==0].query))
overlap = pos_s & neg_s
print(f"[Wikidata] anchor == query pairs  : {same}     (should be 0)")
print(f"[Wikidata] pos/neg overlap        : {len(overlap)}  (should be 0)")
print(f"[Wikidata] balance ratio pos/neg  : "
      f"{(df_wd.label==1).sum()} / {(df_wd.label==0).sum()}")

In [ ]:
save_dataset(df_wd_train, df_wd_val,
             out_dir=OUTPUT_ROOT / "wikidata",
             ds_name="wikidata",
             stats=stats_wd)

In [ ]:
print("=" * 55)
print("  Processing: DBPEDIA")
print("=" * 55)

parsed_dbpedia = parse_records(raw_data["dbpedia"])
print(f"Parsed {len(parsed_dbpedia)} anchors with ≥1 positive")

pairs_db, stats_db = build_balanced_pairs(parsed_dbpedia)

df_db = pd.DataFrame(pairs_db)
df_db["pair_id"] = df_db.apply(
    lambda r: make_pair_id(r.anchor_query, r.query, r.label), axis=1)
df_db = df_db.sample(frac=1, random_state=SEED).reset_index(drop=True)

df_db_train, df_db_val = train_test_split(
    df_db, test_size=VAL_RATIO, random_state=SEED, stratify=df_db["label"])

print(f"\nTotal: {len(df_db)} pairs")
print(f"Train: {len(df_db_train)}  |  Val: {len(df_db_val)}")
print("Label distribution (train):")
print(df_db_train["label"].value_counts().rename({1:"positive(1)", 0:"negative(0)"})
      .to_string())
print("\nNegative sources:")
print(df_db[df_db.label==0]["neg_source"].value_counts().to_string())

In [ ]:
same   = (df_db["anchor_query"] == df_db["query"]).sum()
pos_s  = set(zip(df_db[df_db.label==1].anchor_query, df_db[df_db.label==1].query))
neg_s  = set(zip(df_db[df_db.label==0].anchor_query, df_db[df_db.label==0].query))
overlap = pos_s & neg_s
print(f"[DBpedia]  anchor == query pairs  : {same}     (should be 0)")
print(f"[DBpedia]  pos/neg overlap        : {len(overlap)}  (should be 0)")
print(f"[DBpedia]  balance ratio pos/neg  : "
      f"{(df_db.label==1).sum()} / {(df_db.label==0).sum()}")

In [ ]:
save_dataset(df_db_train, df_db_val,
             out_dir=OUTPUT_ROOT / "dbpedia",
             ds_name="dbpedia",
             stats=stats_db)

In [ ]:
print("\n" + "=" * 60)
print("  FINAL SUMMARY — Two Separate Fine-Tuning Datasets")
print("=" * 60)
summary = pd.DataFrame([
    {
        "Dataset"     : "Wikidata",
        "Total pairs" : len(df_wd),
        "Train"       : len(df_wd_train),
        "Val"         : len(df_wd_val),
        "Pos pairs"   : (df_wd.label==1).sum(),
        "Neg pairs"   : (df_wd.label==0).sum(),
        "Hard negs"   : stats_wd["neg_hard"],
        "Cross negs"  : stats_wd["neg_cross"],
    },
    {
        "Dataset"     : "DBpedia",
        "Total pairs" : len(df_db),
        "Train"       : len(df_db_train),
        "Val"         : len(df_db_val),
        "Pos pairs"   : (df_db.label==1).sum(),
        "Neg pairs"   : (df_db.label==0).sum(),
        "Hard negs"   : stats_db["neg_hard"],
        "Cross negs"  : stats_db["neg_cross"],
    },
])
print(summary.to_string(index=False))
print("\n✅ Done. Two datasets saved under qwen_finetune_dataset/wikidata/ and /dbpedia/")

## How to load each dataset for Qwen fine-tuning

### HuggingFace `datasets` (recommended)
```python
from datasets import load_dataset

# Wikidata model
ds_wikidata = load_dataset("json", data_files={
    "train": "qwen_finetune_dataset/wikidata/train.json",
    "validation": "qwen_finetune_dataset/wikidata/val.json",
})

# DBpedia model
ds_dbpedia = load_dataset("json", data_files={
    "train": "qwen_finetune_dataset/dbpedia/train.json",
    "validation": "qwen_finetune_dataset/dbpedia/val.json",
})
```

### SentenceTransformer `CosineSimilarityLoss`
```python
from sentence_transformers import SentenceTransformer, InputExample
from torch.utils.data import DataLoader
import pandas as pd

def make_dataloader(csv_path, batch_size=32):
    df = pd.read_csv(csv_path)
    examples = [
        InputExample(texts=[r.anchor_query, r.query], label=float(r.label))
        for _, r in df.iterrows()
    ]
    return DataLoader(examples, shuffle=True, batch_size=batch_size)

# Wikidata fine-tuning
train_wd = make_dataloader("qwen_finetune_dataset/wikidata/train.csv")

# DBpedia fine-tuning
train_db = make_dataloader("qwen_finetune_dataset/dbpedia/train.csv")
```

### JSONL for Qwen3-Embedding training script
```bash
# Train Qwen on Wikidata
python train.py \
  --train_file qwen_finetune_dataset/wikidata/train.jsonl \
  --val_file   qwen_finetune_dataset/wikidata/val.jsonl \
  --model_name Qwen/Qwen3-Embedding \
  --output_dir checkpoints/qwen_wikidata

# Train Qwen on DBpedia
python train.py \
  --train_file qwen_finetune_dataset/dbpedia/train.jsonl \
  --val_file   qwen_finetune_dataset/dbpedia/val.jsonl \
  --model_name Qwen/Qwen3-Embedding \
  --output_dir checkpoints/qwen_dbpedia
```